In [ ]:
# ================================================
# 🌟 T5 NEWS SUMMARIZER - 3 Modes (Very Short / Short / Long)
# ================================================

# 1. Install Libraries
!pip install -q transformers torch gradio

# 2. Imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import gradio as gr

print(" Libraries installed!")

# 3. Load Model
tokenizer = AutoTokenizer.from_pretrained("t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f" Model loaded on {device}!")

# 4. Improved Summarization Function
def summarize(text, mode="short"):
    if not text or not text.strip():
        return " Please enter some text to summarize."

    # Better length control
    config = {
        "very_short": {"max_length": 85,  "min_length": 35,  "length_penalty": 2.8},
        "short":      {"max_length": 140, "min_length": 65,  "length_penalty": 2.2},
        "long":       {"max_length": 240, "min_length": 110, "length_penalty": 1.6}
    }

    cfg = config.get(mode.lower(), config["short"])

    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_length=cfg["max_length"],
        min_length=cfg["min_length"],
        num_beams=6,
        length_penalty=cfg["length_penalty"],
        early_stopping=True,
        no_repeat_ngram_size=3,
        do_sample=False
    )

    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

# 5. Beautiful Gradio Interface
with gr.Blocks(title="AI Summarizer", theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <h1 style='text-align: center; color: #1e3a8a; margin-bottom: 8px;'>
         AI Text Summarizer
    </h1>
    <p style='text-align: center; color: #475569;'>
        Summarize any article in 3 different lengths
    </p>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            input_text = gr.Textbox(
                label=" Paste Article Here",
                placeholder="Enter or paste your news/article text...",
                lines=14
            )

            mode = gr.Radio(
                choices=["Very Short", "Short", "Long"],
                value="Short",
                label=" Summary Length"
            )

            btn = gr.Button(" Generate Summary", variant="primary", size="large")

        with gr.Column(scale=1):
            output_text = gr.Textbox(
                label=" Generated Summary",
                lines=14,
                interactive=False,
                show_copy_button=True
            )

    # Examples
    gr.Examples(
        examples=[
            ["The Indian government announced new policies to promote electric vehicles with heavy subsidies..."],
            ["Scientists discovered a new species of octopus near the Mariana Trench with unique abilities..."],
        ],
        inputs=input_text,
        outputs=output_text,
        fn=lambda x: summarize(x, "short"),
        cache_examples=False
    )

    btn.click(
        fn=summarize,
        inputs=[input_text, mode],
        outputs=output_text
    )

# Launch the Interface
demo.launch(share=True, debug=True)

✅ Libraries installed!


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

✅ Model loaded on cpu!


/tmp/ipykernel_3555/3144814359.py:60: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Summarizer", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e66fe9abb91b3d14f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
